# Predicting Student Health Risk — Diverse Multi-Model Ensemble (v2)

**Kaggle Playground Series** | Metric: **Balanced Accuracy** | Classes: `at-risk` (85.9%), `unhealthy` (8.4%), `fit` (5.8%)

## What's new in v2 vs. a basic 3-model stack

Adapted from the strategy in a previous Playground winner's writeup (diverse model zoo, heavy FE, logit-stacking, greedy ensemble pruning, and — most importantly — **trusting CV over the public leaderboard**):

1. **Heavier feature engineering**: adds proper out-of-fold K-Fold target encoding on top of the v1 interaction features (~29 -> ~60+ features)
2. **Model diversity**: LightGBM, XGBoost, CatBoost + ExtraTrees, RandomForest, HistGradientBoosting, and an MLP — deliberately different model families (tree-split-based, bagged, boosted, neural) so their errors are less correlated
3. **Multiple ensemblers compared on OOF CV**: simple average, LR-on-probabilities, **LR-on-logits** (log-odds — often the strongest simple stacker), an MLP meta-learner, and **greedy forward ensemble selection** (Caruana-style — repeatedly adds whichever model improves OOF the most, naturally pruning bloat instead of growing indefinitely)
4. **A CV-trust framework** at the end: don't chase the public LB with blind blends of other people's high-scoring public notebooks — that reliably overfits the public test split and craters on the private LB. Pick your final 2 submissions using **CV-LB agreement**, not raw public LB rank.

Runtime: the tree ensemble is a few minutes per fold on GPU. RF/ET/HGB/MLP are CPU-only and slower — there's a toggle (`INCLUDE_DIVERSE_MODELS`) to skip them for a fast run, or use fewer folds for just that tier.


In [ ]:
# If any of these are missing on your Kaggle image, uncomment:
# !pip install -q lightgbm xgboost catboost

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, os, gc
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import balanced_accuracy_score, classification_report, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.utils.class_weight import compute_sample_weight

import lightgbm as lgb
import xgboost as xgb
import catboost as cb

SEED = 42
N_FOLDS = 7                    # folds for the GPU boosted-tree tier
N_FOLDS_DIVERSE = 5            # fewer folds for the slower CPU tier (RF/ET/HGB/MLP)
INCLUDE_DIVERSE_MODELS = True  # set False to skip RF/ET/HGB/MLP for a much faster run
np.random.seed(SEED)
print('lightgbm', lgb.__version__, '| xgboost', xgb.__version__, '| catboost', cb.__version__)


## 1. Load & (optionally) Blend Data

In [ ]:
COMP_SLUG = 'playground-series-s6e5'   # <-- update to match your competition's URL slug
COMP_DIR = f'/kaggle/input/{COMP_SLUG}/'
if not os.path.exists(COMP_DIR):
    COMP_DIR = '.'

train_df = pd.read_csv(os.path.join(COMP_DIR, 'train.csv'))
test_df  = pd.read_csv(os.path.join(COMP_DIR, 'test.csv'))
print(f'Competition train: {train_df.shape}')
print(f'Competition test:  {test_df.shape}')

TARGET = 'health_condition'
train_full = train_df.copy()

try:
    import kagglehub
    orig_path = kagglehub.dataset_download('ziya07/college-student-health-behavior-dataset')
    orig_files = [f for f in os.listdir(orig_path) if f.endswith('.csv')]
    orig_df = pd.read_csv(os.path.join(orig_path, orig_files[0]))
    common_cols = [c for c in train_full.columns if c in orig_df.columns and c != 'id']
    if TARGET in orig_df.columns and len(common_cols) >= 6:
        orig_df = orig_df[common_cols + [TARGET]].copy()
        orig_df['id'] = range(train_full['id'].max() + 1, train_full['id'].max() + 1 + len(orig_df))
        train_full = pd.concat([train_full, orig_df], axis=0, ignore_index=True)
        print(f'Blended train: {train_full.shape}')
    else:
        print('Original dataset schema does not align closely enough — skipping blend.')
except Exception as e:
    print(f'Original dataset not available/blended (this is fine): {e}')

train_full = train_full.dropna(subset=[TARGET]).reset_index(drop=True)
print(train_full[TARGET].value_counts(normalize=True))


## 2. EDA — Key Findings (recap)

- `stress_level='medium'` -> ~99.4% **at-risk** regardless of activity
- `stress_level='low'` + sedentary/moderate -> ~99.5% **at-risk**
- `stress_level='low'` + active -> splits fit vs at-risk, driven by `sleep_duration`
- `stress_level='high'` -> splits at-risk vs unhealthy, almost never fit
- Missingness is MCAR (no relationship to target) — simple imputation is fine, no need for missing-indicator features


## 3. Feature Engineering — Interactions + Heavy FE via K-Fold Target Encoding

We keep the v1 interaction features (baked-in domain knowledge) and add proper **out-of-fold target encoding**: for each categorical/engineered-categorical column, encode the smoothed probability of each of the 3 classes using only *other* folds — never the row's own fold — to avoid leakage. This is one of the highest-value "heavy FE" techniques for tree models on categorical-heavy tabular data.

In [ ]:
num_cols = ['sleep_duration','heart_rate','bmi','calorie_expenditure','step_count','exercise_duration','water_intake']
cat_cols_raw = ['diet_type','stress_level','sleep_quality','physical_activity_level','smoking_alcohol','gender']

def engineer_features(df):
    df = df.copy()
    for c in num_cols:
        df[c] = df[c].fillna(df[c].median())
    for c in cat_cols_raw:
        df[c] = df[c].astype(str).replace('nan', 'missing')

    df['stress_activity'] = df['stress_level'] + '_' + df['physical_activity_level']
    df['sleep_bin'] = pd.cut(df['sleep_duration'], bins=[-1, 5, 6, 7, 8, 24],
                              labels=['vlow', 'low', 'mid', 'high', 'vhigh']).astype(str)
    df['stress_activity_sleep'] = df['stress_activity'] + '_' + df['sleep_bin']
    df['stress_sleepquality'] = df['stress_level'] + '_' + df['sleep_quality']
    df['bmi_cat'] = pd.cut(df['bmi'], bins=[0, 18.5, 25, 30, 100],
                            labels=['under', 'normal', 'over', 'obese']).astype(str)
    df['bmi_hr_ratio'] = df['bmi'] / (df['heart_rate'] + 1)
    df['calorie_per_step'] = df['calorie_expenditure'] / (df['step_count'] + 1)
    df['exercise_per_step'] = df['exercise_duration'] / (df['step_count'] + 1)
    df['water_per_bmi'] = df['water_intake'] / (df['bmi'] + 1)
    df['activity_intensity'] = df['step_count'] * df['exercise_duration'] / 1000.0
    df['rest_recovery'] = df['sleep_duration'] * df['water_intake']
    df['stress_x_smoking'] = df['stress_level'] + '_' + df['smoking_alcohol']
    df['diet_x_activity'] = df['diet_type'] + '_' + df['physical_activity_level']
    df['hr_x_bmi_cat'] = df['bmi_cat'] + '_' + df['stress_level']

    # additional numeric ratios / polynomial terms — cheap "heavy FE" additions
    df['sleep_x_quality_gap'] = df['sleep_duration'] - df['sleep_duration'].median()
    df['cal_x_exercise'] = df['calorie_expenditure'] * df['exercise_duration'] / 1000.0
    df['step_sq'] = (df['step_count'] / 1000.0) ** 2
    df['bmi_sq'] = df['bmi'] ** 2
    df['hr_sq'] = df['heart_rate'] ** 2
    df['water_x_exercise'] = df['water_intake'] * df['exercise_duration']

    stress_score_map = {'low': 0, 'medium': 1, 'high': 2, 'missing': 1}
    activity_score_map = {'active': 0, 'moderate': 1, 'sedentary': 2, 'missing': 1}
    sleepq_score_map = {'good': 0, 'average': 1, 'poor': 2, 'missing': 1}
    df['risk_score'] = (df['stress_level'].map(stress_score_map).fillna(1)
                         + df['physical_activity_level'].map(activity_score_map).fillna(1)
                         + df['sleep_quality'].map(sleepq_score_map).fillna(1)
                         + (df['sleep_duration'] < 6).astype(int) * 2)
    return df

train_fe = engineer_features(train_full)
test_fe = engineer_features(test_df)

engineered_cat_cols = ['stress_activity', 'sleep_bin', 'stress_activity_sleep', 'stress_sleepquality',
                        'bmi_cat', 'stress_x_smoking', 'diet_x_activity', 'hr_x_bmi_cat']
all_cat_cols = cat_cols_raw + engineered_cat_cols

for col in all_cat_cols:
    combined_cats = pd.concat([train_fe[col], test_fe[col]]).astype(str).unique()
    train_fe[col] = pd.Categorical(train_fe[col].astype(str), categories=combined_cats)
    test_fe[col] = pd.Categorical(test_fe[col].astype(str), categories=combined_cats)

print(f'Base engineered shape -> train: {train_fe.shape}, test: {test_fe.shape}')
print(f'Categorical columns ({len(all_cat_cols)}): {all_cat_cols}')


In [ ]:
# --- Out-of-fold K-Fold target encoding (smoothed, per-class probability) ---
def kfold_target_encode(train_col, y_arr, test_col, n_splits=5, smoothing=20, seed=42):
    # Returns (train_encoded[n,n_classes], test_encoded[n,n_classes]) — leak-safe OOF encoding.
    n_classes = len(np.unique(y_arr))
    global_means = np.array([(y_arr == c).mean() for c in range(n_classes)])
    train_enc = np.zeros((len(train_col), n_classes))
    test_enc = np.zeros((len(test_col), n_classes))
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)

    df_tr = pd.DataFrame({'cat': train_col.astype(str).values})
    df_te = pd.DataFrame({'cat': test_col.astype(str).values})

    for c in range(n_classes):
        target_c = (y_arr == c).astype(float)
        # test encoding uses the FULL training data (safe — test has no labels to leak)
        stats_full = df_tr.assign(t=target_c).groupby('cat')['t'].agg(['mean', 'count'])
        smooth_full = (stats_full['mean'] * stats_full['count'] + global_means[c] * smoothing) / (stats_full['count'] + smoothing)
        test_enc[:, c] = df_te['cat'].map(smooth_full).fillna(global_means[c]).values

        for tr_idx, val_idx in kf.split(df_tr):
            stats = df_tr.iloc[tr_idx].assign(t=target_c[tr_idx]).groupby('cat')['t'].agg(['mean', 'count'])
            smooth = (stats['mean'] * stats['count'] + global_means[c] * smoothing) / (stats['count'] + smoothing)
            train_enc[val_idx, c] = df_tr.iloc[val_idx]['cat'].map(smooth).fillna(global_means[c]).values

    return train_enc, test_enc


le_target = LabelEncoder()
y = le_target.fit_transform(train_fe[TARGET])
class_names = list(le_target.classes_)
N_CLASSES = len(class_names)
print('Classes:', class_names)

te_cols_source = ['stress_level', 'physical_activity_level', 'sleep_quality', 'diet_type',
                   'smoking_alcohol', 'gender', 'bmi_cat', 'stress_activity', 'stress_activity_sleep']

for col in te_cols_source:
    tr_enc, te_enc = kfold_target_encode(train_fe[col], y, test_fe[col], n_splits=5, smoothing=20, seed=SEED)
    for c in range(N_CLASSES):
        train_fe[f'te_{col}_{class_names[c]}'] = tr_enc[:, c]
        test_fe[f'te_{col}_{class_names[c]}'] = te_enc[:, c]

print(f'After target encoding -> train: {train_fe.shape}, test: {test_fe.shape}')


In [ ]:
feature_cols = [c for c in train_fe.columns if c not in ['id', TARGET]]
X_train = train_fe[feature_cols].copy()
X_test = test_fe[feature_cols].copy()
test_ids = test_fe['id'].values

for c in all_cat_cols:
    X_train[c] = X_train[c].astype('category')
    X_test[c] = X_test[c].astype('category')

print(f'Final feature matrix -> X_train: {X_train.shape}, X_test: {X_test.shape} ({len(feature_cols)} features)')

# --- A fully-numeric version for models that can't take pandas 'category' dtype natively (RF/ET/HGB/MLP) ---
X_train_num = X_train.copy()
X_test_num = X_test.copy()
for c in all_cat_cols:
    X_train_num[c] = X_train_num[c].cat.codes
    X_test_num[c] = X_test_num[c].cat.codes

scaler = StandardScaler()
X_train_num_scaled = pd.DataFrame(scaler.fit_transform(X_train_num), columns=X_train_num.columns, index=X_train_num.index)
X_test_num_scaled = pd.DataFrame(scaler.transform(X_test_num), columns=X_test_num.columns, index=X_test_num.index)
print('Numeric (scaled) matrices ready for RF/ET/HGB/MLP.')


## 4. Model Configs (GPU-Accelerated Boosted Trees)

In [ ]:
def lgb_balanced_acc(y_pred, dataset):
    y_true = dataset.get_label().astype(int)
    y_pred = y_pred.reshape(N_CLASSES, -1).T if y_pred.ndim == 1 else y_pred
    return 'balanced_acc', balanced_accuracy_score(y_true, y_pred.argmax(axis=1)), True

GPU_AVAILABLE = True

lgb_params = dict(
    objective='multiclass', num_class=N_CLASSES,
    device='gpu' if GPU_AVAILABLE else 'cpu',
    n_estimators=3000, learning_rate=0.03, num_leaves=127, max_depth=-1,
    min_child_samples=30, subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=0.5, class_weight='balanced', random_state=SEED, verbose=-1
)
xgb_params = dict(
    objective='multi:softprob', num_class=N_CLASSES,
    device='cuda' if GPU_AVAILABLE else 'cpu', tree_method='hist', enable_categorical=True,
    n_estimators=3000, learning_rate=0.03, max_depth=8, subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=1.0, eval_metric='mlogloss', early_stopping_rounds=150, random_state=SEED
)
cat_params = dict(
    loss_function='MultiClass', task_type='GPU' if GPU_AVAILABLE else 'CPU',
    iterations=3000, learning_rate=0.03, depth=8, l2_leaf_reg=3.0,
    auto_class_weights='Balanced', cat_features=all_cat_cols, random_seed=SEED,
    od_type='Iter', od_wait=150, verbose=0
)

try:
    _test_model = lgb.LGBMClassifier(device='gpu', n_estimators=5, verbose=-1)
    _test_model.fit(X_train.iloc[:500], y[:500], categorical_feature=all_cat_cols)
    print('GPU LightGBM OK')
except Exception as e:
    print(f'GPU not available ({e}) -> falling back to CPU')
    GPU_AVAILABLE = False
    lgb_params['device'] = 'cpu'; xgb_params['device'] = 'cpu'; cat_params['task_type'] = 'CPU'


## 5. Train the Boosted-Tree Tier (LightGBM, XGBoost, CatBoost)

In [ ]:
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
sample_weights_full = compute_sample_weight('balanced', y)

oof_preds = {}
test_preds = {}
fitted_lgb_models = []

for name in ['lgb', 'xgb', 'cat']:
    oof_preds[name] = np.zeros((len(X_train), N_CLASSES))
    test_preds[name] = np.zeros((len(X_test), N_CLASSES))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y)):
    print(f'\n=== Boosted-tree fold {fold + 1}/{N_FOLDS} ===')
    xt, xv = X_train.iloc[tr_idx], X_train.iloc[val_idx]
    yt, yv = y[tr_idx], y[val_idx]
    wt = sample_weights_full[tr_idx]

    lgb_train = lgb.Dataset(xt, label=yt, categorical_feature=all_cat_cols, weight=wt, free_raw_data=False)
    lgb_val = lgb.Dataset(xv, label=yv, categorical_feature=all_cat_cols, reference=lgb_train, free_raw_data=False)
    booster = lgb.train(lgb_params, lgb_train, valid_sets=[lgb_val], feval=lgb_balanced_acc,
                         num_boost_round=lgb_params['n_estimators'],
                         callbacks=[lgb.early_stopping(150, verbose=False), lgb.log_evaluation(0)])
    oof_preds['lgb'][val_idx] = booster.predict(xv, num_iteration=booster.best_iteration)
    test_preds['lgb'] += booster.predict(X_test, num_iteration=booster.best_iteration) / N_FOLDS
    fitted_lgb_models.append(booster)
    lgb_fold_score = balanced_accuracy_score(yv, oof_preds['lgb'][val_idx].argmax(1))
    print(f'  lgb: {lgb_fold_score:.5f}')

    xgb_model = xgb.XGBClassifier(**xgb_params)
    xgb_model.fit(xt, yt, sample_weight=wt, eval_set=[(xv, yv)], verbose=False)
    oof_preds['xgb'][val_idx] = xgb_model.predict_proba(xv)
    test_preds['xgb'] += xgb_model.predict_proba(X_test) / N_FOLDS
    xgb_fold_score = balanced_accuracy_score(yv, oof_preds['xgb'][val_idx].argmax(1))
    print(f'  xgb: {xgb_fold_score:.5f}')

    cat_model = cb.CatBoostClassifier(**cat_params)
    cat_model.fit(xt, yt, eval_set=(xv, yv), use_best_model=True)
    oof_preds['cat'][val_idx] = cat_model.predict_proba(xv)
    test_preds['cat'] += cat_model.predict_proba(X_test) / N_FOLDS
    cat_fold_score = balanced_accuracy_score(yv, oof_preds['cat'][val_idx].argmax(1))
    print(f'  cat: {cat_fold_score:.5f}')

    gc.collect()

print('\n=== Boosted-tree OOF Balanced Accuracy ===')
for name in ['lgb', 'xgb', 'cat']:
    print(f'  {name}: {balanced_accuracy_score(y, oof_preds[name].argmax(1)):.5f}')


## 6. Diverse Model Tier — ExtraTrees, RandomForest, HistGradientBoosting, MLP

Different model families reduce correlated errors, which is exactly what a stacker needs to add real value beyond the best single model. This tier runs on CPU using the numeric/scaled feature matrix. Toggle `INCLUDE_DIVERSE_MODELS = False` above to skip it.

In [ ]:
if INCLUDE_DIVERSE_MODELS:
    skf_d = StratifiedKFold(n_splits=N_FOLDS_DIVERSE, shuffle=True, random_state=SEED)

    diverse_configs = {
        'rf': RandomForestClassifier(n_estimators=400, max_depth=18, min_samples_leaf=5,
                                      class_weight='balanced', n_jobs=-1, random_state=SEED),
        'et': ExtraTreesClassifier(n_estimators=400, max_depth=20, min_samples_leaf=5,
                                    class_weight='balanced', n_jobs=-1, random_state=SEED),
        'hgb': HistGradientBoostingClassifier(max_iter=400, max_depth=8, learning_rate=0.06,
                                               l2_regularization=0.5, random_state=SEED),
        'mlp': MLPClassifier(hidden_layer_sizes=(128, 64), activation='relu', alpha=1e-4,
                              learning_rate_init=1e-3, max_iter=150, early_stopping=True,
                              n_iter_no_change=10, random_state=SEED),
    }

    for name in diverse_configs:
        oof_preds[name] = np.zeros((len(X_train), N_CLASSES))
        test_preds[name] = np.zeros((len(X_test), N_CLASSES))

    sample_weights_full_num = compute_sample_weight('balanced', y)

    for fold, (tr_idx, val_idx) in enumerate(skf_d.split(X_train_num_scaled, y)):
        print(f'\n=== Diverse-tier fold {fold + 1}/{N_FOLDS_DIVERSE} ===')
        xt, xv = X_train_num_scaled.iloc[tr_idx], X_train_num_scaled.iloc[val_idx]
        yt, yv = y[tr_idx], y[val_idx]
        wt = sample_weights_full_num[tr_idx]

        for name, base_model in diverse_configs.items():
            from sklearn.base import clone
            model = clone(base_model)
            if name == 'mlp':
                model.fit(xt, yt)  # MLPClassifier doesn't take sample_weight
            elif name == 'hgb':
                model.fit(xt, yt, sample_weight=wt)
            else:
                model.fit(xt, yt)  # class_weight='balanced' already set for rf/et
            oof_preds[name][val_idx] = model.predict_proba(xv)
            test_preds[name] += model.predict_proba(X_test_num_scaled) / N_FOLDS_DIVERSE
            print(f'  {name}: {balanced_accuracy_score(yv, oof_preds[name][val_idx].argmax(1)):.5f}')
        gc.collect()

    print('\n=== Diverse-tier OOF Balanced Accuracy ===')
    for name in diverse_configs:
        print(f'  {name}: {balanced_accuracy_score(y, oof_preds[name].argmax(1)):.5f}')
else:
    print('INCLUDE_DIVERSE_MODELS is False — skipping RF/ET/HGB/MLP tier.')

model_names = list(oof_preds.keys())
print('\nAll models in the pool:', model_names)


## 7. Compare Ensemblers on OOF CV — Average, LR-on-Probs, LR-on-Logits, MLP Meta, Greedy Selection

We compute several ensembling strategies **all measured out-of-fold**, and only trust CV numbers when deciding what to submit. `LR-on-logits` converts each model's probabilities to log-odds before fitting the meta-learner — this often outperforms feeding raw probabilities directly, since logits are unbounded and linear-friendlier.

In [ ]:
def to_logits(probs, eps=1e-6):
    p = np.clip(probs, eps, 1 - eps)
    return np.log(p / (1 - p))

oof_stack_probs = np.hstack([oof_preds[n] for n in model_names])
test_stack_probs = np.hstack([test_preds[n] for n in model_names])
oof_stack_logits = np.hstack([to_logits(oof_preds[n]) for n in model_names])
test_stack_logits = np.hstack([to_logits(test_preds[n]) for n in model_names])

results = {}

# 1) Simple average
avg_oof = np.mean([oof_preds[n] for n in model_names], axis=0)
avg_test = np.mean([test_preds[n] for n in model_names], axis=0)
results['simple_average'] = (balanced_accuracy_score(y, avg_oof.argmax(1)), avg_oof, avg_test)

# 2) LR on raw probabilities
skf_meta = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
lr_probs_oof = np.zeros((len(y), N_CLASSES))
for tr_idx, val_idx in skf_meta.split(oof_stack_probs, y):
    m = LogisticRegression(class_weight='balanced', max_iter=3000, C=1.0, random_state=SEED)
    m.fit(oof_stack_probs[tr_idx], y[tr_idx])
    lr_probs_oof[val_idx] = m.predict_proba(oof_stack_probs[val_idx])
lr_probs_final = LogisticRegression(class_weight='balanced', max_iter=3000, C=1.0, random_state=SEED)
lr_probs_final.fit(oof_stack_probs, y)
results['lr_on_probs'] = (balanced_accuracy_score(y, lr_probs_oof.argmax(1)), lr_probs_oof,
                           lr_probs_final.predict_proba(test_stack_probs))

# 3) LR on logits ("LR-Logits" — cdeotte-style stacker)
lr_logits_oof = np.zeros((len(y), N_CLASSES))
for tr_idx, val_idx in skf_meta.split(oof_stack_logits, y):
    m = LogisticRegression(class_weight='balanced', max_iter=3000, C=0.5, random_state=SEED)
    m.fit(oof_stack_logits[tr_idx], y[tr_idx])
    lr_logits_oof[val_idx] = m.predict_proba(oof_stack_logits[val_idx])
lr_logits_final = LogisticRegression(class_weight='balanced', max_iter=3000, C=0.5, random_state=SEED)
lr_logits_final.fit(oof_stack_logits, y)
results['lr_on_logits'] = (balanced_accuracy_score(y, lr_logits_oof.argmax(1)), lr_logits_oof,
                            lr_logits_final.predict_proba(test_stack_logits))

# 4) Small MLP meta-learner on logits
mlp_meta_oof = np.zeros((len(y), N_CLASSES))
for tr_idx, val_idx in skf_meta.split(oof_stack_logits, y):
    m = MLPClassifier(hidden_layer_sizes=(32,), alpha=1e-3, max_iter=300,
                       early_stopping=True, n_iter_no_change=15, random_state=SEED)
    m.fit(oof_stack_logits[tr_idx], y[tr_idx])
    mlp_meta_oof[val_idx] = m.predict_proba(oof_stack_logits[val_idx])
mlp_meta_final = MLPClassifier(hidden_layer_sizes=(32,), alpha=1e-3, max_iter=300,
                                early_stopping=True, n_iter_no_change=15, random_state=SEED)
mlp_meta_final.fit(oof_stack_logits, y)
results['mlp_meta'] = (balanced_accuracy_score(y, mlp_meta_oof.argmax(1)), mlp_meta_oof,
                        mlp_meta_final.predict_proba(test_stack_logits))

print('=== Ensembler comparison (OOF Balanced Accuracy) ===')
for name, (score, _, _) in sorted(results.items(), key=lambda kv: -kv[1][0]):
    print(f'  {name:16s}: {score:.6f}')


## 8. Greedy Forward Ensemble Selection (Caruana-style)

Rather than always averaging *every* model — which the winner's writeup notes can actually hurt past a certain ensemble size — we greedily add whichever model (with replacement, so a strong model can be weighted more than once) improves OOF balanced accuracy the most, and stop when nothing helps anymore. This naturally finds a compact, high-performing subset/weighting instead of over-growing the ensemble.

In [ ]:
def greedy_ensemble_selection(oof_dict, test_dict, y_true, max_iters=60, tol=1e-6):
    names = list(oof_dict.keys())
    picked = []
    current_sum_oof = np.zeros_like(next(iter(oof_dict.values())))
    current_sum_test = np.zeros_like(next(iter(test_dict.values())))
    best_score = -1.0
    history = []

    for _ in range(max_iters):
        best_name, best_trial_score = None, best_score
        for n in names:
            trial_oof = (current_sum_oof + oof_dict[n]) / (len(picked) + 1)
            s = balanced_accuracy_score(y_true, trial_oof.argmax(1))
            if s > best_trial_score + tol:
                best_trial_score, best_name = s, n
        if best_name is None:
            break
        picked.append(best_name)
        current_sum_oof += oof_dict[best_name]
        current_sum_test += test_dict[best_name]
        best_score = best_trial_score
        history.append((best_name, best_score))

    final_oof = current_sum_oof / max(len(picked), 1)
    final_test = current_sum_test / max(len(picked), 1)
    return picked, best_score, final_oof, final_test, history

greedy_picked, greedy_score, greedy_oof, greedy_test, greedy_history = greedy_ensemble_selection(
    oof_preds, test_preds, y, max_iters=60
)
results['greedy_selection'] = (greedy_score, greedy_oof, greedy_test)

print('Greedy selection order (model added at each step):')
for i, (n, s) in enumerate(greedy_history):
    print(f'  step {i+1:2d}: +{n:6s} -> OOF balanced acc = {s:.6f}')
print(f'\nFinal greedy ensemble ({len(greedy_picked)} picks from {len(set(greedy_picked))} unique models): {greedy_score:.6f}')
from collections import Counter
print('Model weights (count of times picked):', dict(Counter(greedy_picked)))


## 9. Pick the Best Strategy by CV, Then Optimize Per-Class Weights

**This is the point to slow down and trust the numbers, not the public leaderboard.** Pick whichever ensembler scored best OOF above; then apply the same per-class probability-weight optimization as before (grid search on OOF only) since balanced accuracy = mean of per-class recalls.

In [ ]:
best_strategy = max(results.items(), key=lambda kv: kv[1][0])
print(f'Best strategy by OOF CV: {best_strategy[0]} ({best_strategy[1][0]:.6f})')
best_oof_probs = best_strategy[1][1]
final_test_probs = best_strategy[1][2]

def weighted_score(probs, weights, y_true):
    adjusted = probs * np.asarray(weights)[np.newaxis, :]
    return balanced_accuracy_score(y_true, adjusted.argmax(axis=1))

base_score = balanced_accuracy_score(y, best_oof_probs.argmax(1))
best_w = [1.0, 1.0, 1.0]
best_adj_score = base_score

coarse = np.arange(0.4, 3.01, 0.2)
for w1 in coarse:
    for w2 in coarse:
        w = [1.0, w1, w2]
        s = weighted_score(best_oof_probs, w, y)
        if s > best_adj_score:
            best_adj_score, best_w = s, w

w1c, w2c = best_w[1], best_w[2]
fine = np.arange(-0.18, 0.19, 0.02)
for d1 in fine:
    for d2 in fine:
        w = [1.0, max(0.05, w1c + d1), max(0.05, w2c + d2)]
        s = weighted_score(best_oof_probs, w, y)
        if s > best_adj_score:
            best_adj_score, best_w = s, w

print(f'Best class weights ({class_names[0]}={best_w[0]:.2f}, {class_names[1]}={best_w[1]:.2f}, {class_names[2]}={best_w[2]:.2f})')
print(f'Final OOF balanced accuracy after weighting: {best_adj_score:.6f} (raw: {base_score:.6f})')


## 10. Analysis — Confusion Matrix & Feature Importance

In [ ]:
adjusted_oof = best_oof_probs * np.array(best_w)[np.newaxis, :]
oof_final = adjusted_oof.argmax(axis=1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
cm = confusion_matrix(y, oof_final, normalize='true')
sns.heatmap(cm, annot=True, fmt='.3f', cmap='Blues', xticklabels=class_names, yticklabels=class_names, ax=axes[0])
axes[0].set_title(f'Normalized Confusion Matrix\nBalanced Accuracy: {best_adj_score:.5f}')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')

recalls = cm.diagonal()
axes[1].bar(class_names, recalls, color=['#3498db', '#e74c3c', '#2ecc71'])
axes[1].set_title('Per-Class Recall'); axes[1].set_ylabel('Recall'); axes[1].set_ylim(0, 1.0)
for i, r in enumerate(recalls):
    axes[1].text(i, r + 0.01, f'{r:.4f}', ha='center', fontweight='bold')
plt.tight_layout(); plt.show()

print(classification_report(y, oof_final, target_names=class_names))


In [ ]:
importances = np.mean([m.feature_importance(importance_type='gain') for m in fitted_lgb_models], axis=0)
fi = pd.DataFrame({'feature': feature_cols, 'importance': importances}).sort_values('importance', ascending=True).tail(30)
fig, ax = plt.subplots(figsize=(10, 10))
ax.barh(fi['feature'], fi['importance'], color='steelblue')
ax.set_title('Top 30 LightGBM Feature Importances (gain, avg over folds)')
plt.tight_layout(); plt.show()


## 11. Generate Submission

In [ ]:
final_adjusted = final_test_probs * np.array(best_w)[np.newaxis, :]
final_preds = final_adjusted.argmax(axis=1)
final_labels = le_target.inverse_transform(final_preds)

submission = pd.DataFrame({'id': test_ids, TARGET: final_labels})
submission.to_csv('submission.csv', index=False)
print('Submission shape:', submission.shape)
print(submission[TARGET].value_counts(normalize=True))
print(submission.head(10))


## 12. Trust Your CV — Final Submission Selection Strategy

A recurring lesson from Playground Series winners: **once the public leaderboard fills up with "blind blending" notebooks (people mixing high-scoring public kernels without a real CV framework), public LB rank stops being a reliable signal.** Chasing it usually means overfitting to the public test split, which then collapses on the private LB.

Practical checklist before your final submissions:

1. **Only compare models/ensembles using OOF CV scores computed with a consistent, leak-free fold scheme** (as done throughout this notebook) — not by re-uploading and watching the public LB move.
2. If you *do* try blending with a public high-scoring notebook out of curiosity, **don't let it become one of your 2 final submissions** unless your own CV also independently supports it.
3. Kaggle lets you pick **2 final submissions**. A robust pattern used by past winners:
   - One submission: your **highest-CV** ensemble overall.
   - Another submission: among your submissions with a *reasonable* public LB score, the one with the **highest CV** (i.e., best CV-LB agreement), not the single highest public LB score.
4. Watch for a plateau: if adding more models to the ensemble stops improving (or starts hurting) OOF CV, stop growing it — that's what the greedy selection step above does automatically, and it mirrors the "ensemble bloat past ~100 models started deteriorating" pattern winners have reported.
5. If your public LB rank drifts down while your CV keeps improving, that is not necessarily bad news — it can mean the public leaderboard is currently dominated by overfit blends rather than genuinely better models.
